In [ ]:
# нагрузочное тестирование локальной onnx-модели ориентации на калининградских эссе
ONNX_PATH = "/workspace/HDD/handwriting-recognition/page_orientation/checkpoints_1/page_orientation.onnx"
DATA_DIR  = "/mnt/DATA2/dorin/handwriting-recognition/data/kaliningrad_essays"   # путь к картинкам (рекурсивно)
N_WARMUP  = 5

In [ ]:
# модель и препроцессинг (тот же, что в обучении: Resize(511, max_size=512) + CenterCrop(512))
import time
from pathlib import Path

import numpy as np
import torch
import onnxruntime as ort
from PIL import Image, ImageFile
from torchvision import transforms as T

ImageFile.LOAD_TRUNCATED_IMAGES = True
CLASSES = [0, -90, 180, 90]

transform = T.Compose([
    T.Resize(511, max_size=512),
    T.CenterCrop(512),
    T.ToTensor(),
    T.Normalize(mean=[0.694, 0.695, 0.693], std=[0.299, 0.296, 0.301]),
])

def preprocess(img):
    return transform(img).unsqueeze(0).numpy()

sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
INPUT = sess.get_inputs()[0].name

def predict(img):
    logits = sess.run(None, {INPUT: preprocess(img)})[0][0]
    i = int(logits.argmax())
    return CLASSES[i]

paths = sorted(p for p in Path(DATA_DIR).rglob("*")
               if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png"))
print(f"модель: {ONNX_PATH}")
print(f"картинок найдено: {len(paths)}")

In [ ]:
# нагрузочный прогон: все картинки, время препроцесса и инференса отдельно
from collections import Counter
from tqdm import tqdm

pre_ms, inf_ms, total_ms = [], [], []
angles = Counter()

# прогрев: первые вызовы ORT медленнее из-за аллокаций
for p in paths[:N_WARMUP]:
    predict(Image.open(p).convert("RGB"))

for p in tqdm(paths):
    img = Image.open(p).convert("RGB")

    t0 = time.perf_counter()
    x = preprocess(img)
    t1 = time.perf_counter()
    logits = sess.run(None, {INPUT: x})[0][0]
    t2 = time.perf_counter()

    pre_ms.append((t1 - t0) * 1000)
    inf_ms.append((t2 - t1) * 1000)
    total_ms.append((t2 - t0) * 1000)
    angles[CLASSES[int(logits.argmax())]] += 1

pre_ms, inf_ms, total_ms = map(np.array, (pre_ms, inf_ms, total_ms))

def stats(a):
    return f"p50={np.percentile(a,50):6.1f}  p95={np.percentile(a,95):6.1f}  p99={np.percentile(a,99):6.1f}  mean={a.mean():6.1f}"

print(f"страниц: {len(paths)}")
print(f"препроцесс, мс: {stats(pre_ms)}")
print(f"инференс,   мс: {stats(inf_ms)}")
print(f"итого,      мс: {stats(total_ms)}")
print(f"пропускная способность: {1000 / total_ms.mean():.1f} стр/с (1 поток)")
print(f"предсказанные углы: {dict(angles)}")

In [ ]:
# графики: распределение полной латентности и разбивка препроцесс/инференс
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(total_ms, bins=40)
ax[0].axvline(np.percentile(total_ms, 95), color="r", ls="--", label="p95")
ax[0].set_xlabel("мс на страницу"); ax[0].set_title("полная латентность"); ax[0].legend()

ax[1].bar(["препроцесс", "инференс"], [np.mean(pre_ms), np.mean(inf_ms)])
ax[1].set_ylabel("мс, среднее"); ax[1].set_title("из чего складывается")
plt.tight_layout(); plt.show()